# 06 평가 ③ 시험 사진 예측 저장 (Colab, **GPU 필요: 런타임 → T4**)
Food-101 **공식 시험 사진 25,250장**(학습에 안 쓴 사진)을 앱과 같은 ConvNeXt에 넣고, 사진마다 **상위 5개 음식과 확률**을 저장한다.
이 파일로 `evaluation/merge_eval.py`가 "1등만 믿기"와 "후보 합치기" 중 어느 쪽이 알레르기 경고를 덜 놓치는지 채점한다 (`docs/역할분담.md` 서준영 ③).

| 파트 | 하는 일 |
|---|---|
| 0 | 드라이브 연결 · 코드 받기 |
| 1 | Food-101 시험 사진 풀기 (5~10분) |
| 2 | ConvNeXt 예측 → `찍먹_실험기록/표/convnext_test_top5.csv`, 내 컴퓨터로도 내려받기 |

처음부터 끝까지 15분 안팎. **실행:** 런타임 → 런타임 유형 변경 → T4 GPU → 런타임 → 모두 실행.

## 0. 드라이브 연결 · 코드 받기 (04 첫 셀과 같음)

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
EXP = find(TEAM, '찍먹_실험기록')
OUT = f'{EXP}/표'
os.makedirs(OUT, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
assert torch.cuda.is_available(), '런타임 → 런타임 유형 변경 → T4 GPU'
dev = 'cuda'
print('준비 완료', torch.cuda.get_device_name())

## 1. Food-101 시험 사진 풀기
- 드라이브 `data1/food-101.tar.gz`를 Colab 안에 푼다 (04의 7번과 같은 방법). 이미 풀려 있으면 건너뛴다.
- 시험 사진 목록은 공식 `meta/test.txt`다 (101종 × 250장). ConvNeXt 학습에 한 장도 안 쓴 사진이다.

In [ ]:
F101 = '/content/food-101'
if not os.path.exists(f'{F101}/meta/test.txt'):
    t = time.time()
    subprocess.run(['tar', 'xzf', f'{DATA}/food-101.tar.gz', '-C', '/content'], check=True)
    print(f'압축 풀기 {time.time() - t:.0f}초')
test = open(f'{F101}/meta/test.txt').read().split()
print(f'시험 사진 {len(test):,}장, 음식 {len({p.split("/")[0] for p in test})}종')

## 2. ConvNeXt 예측
- 앱과 **같은 모델**(`찍먹_실험기록/best/convnext_base_exp004/best.pt`, 한솔 최종 모델)·**같은 전처리**(`model.preprocess`)·**같은 보정**(`model.temperature`)을 쓴다. 확률은 보정된 값이라 `judge.py` 기준값(0.95/0.6)과 그대로 비교된다.
- 상위 5개를 저장한다. 합치기는 3개만 쓰지만, 나중에 다른 실험에도 쓸 수 있게 5개를 남긴다.
- 끝나면 top-1 정확도를 찍는다. `docs/진행상황.md`의 0.888(음식당 20장)과 비슷하면 정상이다.
- 저장 파일: `path`(음식/사진번호), `true`(실제 음식), `c1`~`c5`(예측 음식), `p1`~`p5`(확률)

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from modeling.model import load_model, TF

class TestPhotos(Dataset):
    def __len__(self):
        return len(test)
    def __getitem__(self, i):
        return model.preprocess(Image.open(f'{F101}/images/{test[i]}.jpg').convert('RGB'))

model, classes = load_model(f'{EXP}/best/convnext_base_exp004/best.pt', f'{REPO}/modeling/classes.txt')
print('모델:', model.kind, '| 보정 T =', model.temperature)
model = model.to(dev)

t = time.time()
probs, idxs = [], []
with torch.no_grad():
    for x in DataLoader(TestPhotos(), batch_size=128, num_workers=2):
        p, i = (model(x.to(dev)) / model.temperature).softmax(1).topk(5)
        probs.append(p.cpu()); idxs.append(i.cpu())
probs, idxs = torch.cat(probs).numpy(), torch.cat(idxs).numpy()

df = pd.DataFrame({'path': test, 'true': [p.split('/')[0] for p in test]})
for k in range(5):
    df[f'c{k + 1}'] = [classes[j] for j in idxs[:, k]]
    df[f'p{k + 1}'] = probs[:, k].round(5)
path = f'{OUT}/convnext_test_top5_exp004.csv'   # 예전 Tiny 결과는 그대로 둠
df.to_csv(path, index=False, encoding='utf-8-sig')

top5 = df[[f'c{k}' for k in range(1, 6)]].eq(df.true, axis=0).any(axis=1).mean()
print(f'{len(df):,}장 {time.time() - t:.0f}초 | top-1 {(df.c1 == df.true).mean():.3f}, top-5 {top5:.3f}')
print('저장 →', path)

from google.colab import files
files.download(path)   # 내 컴퓨터 '다운로드' 폴더로

## 끝
- 내 컴퓨터 **다운로드 폴더**에 `convnext_test_top5_exp004.csv`가 받아진다. 안 받아졌으면 드라이브 `찍먹_실험기록/표/`에서 내려받는다.
- Claude에게 **"06 다 돌렸어"**라고 말하면 `python -m evaluation.merge_eval <파일>`로 채점해서 `evaluation/합치기_결과.md`를 만든다.